average delivery time 

%of orders arrived late by state

do late deliveries get lower review?

order peaks at?which day of the week and what hours



In [55]:
import pandas as pd
import matplotlib.pyplot as  plt

In [56]:
reviews = pd.read_csv("olist_order_reviews_dataset.csv")
orders = pd.read_csv("olist_orders_dataset.csv")
customers = pd.read_csv("olist_customers_dataset.csv")

In [57]:
##converting dates into datatime format
orders["order_purchase_timestamp"]= pd.to_datetime(orders["order_purchase_timestamp"])
orders["order_approved_at"] = pd.to_datetime(orders["order_approved_at"])
orders["order_delivered_carrier_date"] = pd.to_datetime(orders["order_delivered_carrier_date"])
orders["order_delivered_customer_date"] =  pd.to_datetime(orders["order_delivered_customer_date"])
orders["order_estimated_delivery_date"] = pd.to_datetime(orders["order_estimated_delivery_date"])
print(orders.info())

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
dtypes: datetime64[us](5), str(3)
memory usage: 6.1 MB
None


In [58]:
##average delivery time 
##filtering order that are delivered
delivered_orders= orders[orders['order_status'] == 'delivered']
##finding days took to deliver a order
delivered_orders['delivery_time'] = (delivered_orders['order_delivered_customer_date']-delivered_orders['order_purchase_timestamp']).dt.days
###average delivery days
avg_deliverytime = delivered_orders['delivery_time'].mean().__round__(2)
print(f"average delivery time is:{avg_deliverytime} Days")
print(customers.info())

average delivery time is:12.09 Days
<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: int64(1), str(4)
memory usage: 3.8 MB
None


In [59]:
##percent of order arrived late by state
#filtering orders that are late then estimate delivery date
late_order = delivered_orders[delivered_orders['order_delivered_customer_date'] > delivered_orders['order_estimated_delivery_date']]
#####adding customer state column from customers data set
late_order = late_order.merge(customers[['customer_id','customer_state']],on='customer_id',how='left')
##changing delivery time into how many days late delivery
late_order['delivery_time'] =(late_order['order_delivered_customer_date'] - late_order['order_estimated_delivery_date']).dt.days
# print(late_order.head(3))
# print(late_order['delivery_time'])
#GROUPING late orders acc to state
sw_lateorders = late_order.groupby("customer_state").size()
##adding state in delivered order to find percentage
delivered_orders = delivered_orders.merge(
    customers[['customer_id', 'customer_state']],
    on='customer_id',
    how='left'
)
###grouping total order delivered by state
total_orders = delivered_orders.groupby('customer_state').size()
#finding percentage
late_percentage = (sw_lateorders/ total_orders) * 100
print(late_percentage)

customer_state
AC     3.750000
AL    23.929471
AM     4.137931
AP     4.477612
BA    14.035627
CE    15.324472
DF     7.067308
ES    12.230576
GO     8.175779
MA    19.665272
MG     5.610358
MS    11.554922
MT     6.772009
PA    12.367865
PB    11.025145
PE    10.797238
PI    15.966387
PR     4.996953
RJ    13.473684
RN    10.759494
RO     2.880658
RR    12.195122
RS     7.146866
SC     9.757473
SE    15.223881
SP     5.893682
TO    12.773723
dtype: float64


In [60]:
##do late deliveries get lower reviews?
##mergin review with  late orders``
late_order = late_order.merge(reviews[['order_id','review_score']],on='order_id',how='left')
#getting orders that are delivered early or on time
order_ontime = delivered_orders[delivered_orders['order_delivered_customer_date'] <= delivered_orders['order_estimated_delivery_date']]
##adding review in this data frame
order_ontime =order_ontime.merge(reviews[['order_id','review_score']],on='order_id',how='left')
##comparing
# Average review score for late deliveries
late_avg_review = late_order['review_score'].mean()

# Average review score for on-time or early deliveries
ontime_avg_review = order_ontime['review_score'].mean()

print(f"Average review score for late deliveries: {late_avg_review:.2f}")
print(f"Average review score for on-time/early deliveries: {ontime_avg_review:.2f}")
print("final verdict:late deliveries get lower ratings then on time or early deliveries")

Average review score for late deliveries: 2.57
Average review score for on-time/early deliveries: 4.29
final verdict:late deliveries get lower ratings then on time or early deliveries


In [ ]:
##orders peak at
##finding at which day most orders come
mostorderedday = delivered_orders.groupby(delivered_orders['order_purchase_timestamp'].dt.day_name()).size().sort_values(ascending=False)
print(f"Day in which most order were purchased:{mostorderedday.first_valid_index()} with {mostorderedday.max()} orders.")
#orders peak time
peakhour = delivered_orders.groupby(delivered_orders['order_purchase_timestamp'].dt.hour).size().sort_values(ascending=False)
print(f"Peak ordering hour: {peakhour.index[0]}")
print(f"Number of orders: {peakhour.iloc[0]}")

Day in which most order were purchased:Monday with 15701 orders.
Peak ordering hour: 16
Number of orders: 6476
